# Notebook 03: Faithfulness on Real Data

**Purpose**: Compute internal-consistency faithfulness rho(pi_self, pi_behav) on real datasets — answers RQ3's internal-consistency component.

**Conditions evaluated (3 only)**: Random-k, best-performing protocol from Notebook 02, and (later, Week 8) SATA if real-data transfer works.

## Why this notebook exists (thesis framing)

Accuracy alone can't tell us whether a model is generalising *for the right reasons* — that's the central move the lit review makes in §2.5, going from **invariance** (does the decision rule stay stable across environments?) to **faithfulness** (does the model's stated or measured feature reliance match what actually drives its predictions?). A model can be invariant yet unfaithful (consistently wrong feature, every environment) or faithful yet non-invariant (correctly shifts reliance as the environment shifts). This project's evaluation targets faithfulness specifically because the intervention — demonstration design — operates on a *frozen* model: nothing about the LLM's internal decision rule can be retrained, only which features it's nudged to attend to via which demonstrations it sees.

**RQ3** asks: do configurations that improve OOD accuracy also improve faithfulness, or can accuracy gains coexist with continued reliance on spurious features? This is not a foregone conclusion — Turpin et al. (2023) showed chain-of-thought explanations can be systematically unfaithful (the model changes its answer to match a bias but never mentions the bias in its stated reasoning), and STaDS (Li et al. 2025) found frontier LLMs can be highly *accurate* yet globally *unfaithful* on tabular tasks. RQ3 succeeds specifically if there exist configurations where accuracy improves but ρ(π_self, π_behav) doesn't — that would confirm predictive gains and faithful reliance are genuinely separable outcomes, not the same thing measured twice.

**Why only 3 conditions here, not all 7 from Notebook 02?** This notebook's per-condition compute cost is dominated by the leave-one-out ablation (Step 2), which reruns inference once per feature per query. Running all 7 conditions at that cost isn't affordable within the project's timeline, so the spec narrows to the two conditions most informative for RQ3: random (the no-design baseline) and whichever protocol performed best in Notebook 02 (the condition most likely to show an accuracy/faithfulness split, if one exists).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Step 1: Elicit pi_self (self-reported feature ranking)

Prompt the LLM once per (dataset, condition, seed) via `src/inference/prompts.py::build_feature_ranking_prompt`; parse with `src/evaluation/faithfulness.py::parse_feature_ranking`.

**This is a *global* faithfulness measure, not an instance-level one.** The chain-of-thought faithfulness literature (Turpin et al. 2023; Lanham et al. 2023) asks whether a *single prediction's* stated reasoning matches the computation that produced *that* prediction. STaDS (Li et al. 2025) introduces a complementary, domain-level notion instead: does the model's self-reported feature ranking *for the task as a whole* correspond to its *behavioural* feature ranking, computed independently via ablation (Step 2)? π_self here is elicited once per (dataset, condition, seed) — not per query — because it's a claim about the task ("which features matter for this kind of prediction"), not about any individual row.

In [2]:
import json

import numpy as np
import pandas as pd

from src.data.tableshift_loader import SELECTED_DATASETS
from src.inference.llm_runner import VLLMWorkerRunner
from src.inference.prompts import build_feature_ranking_prompt
from src.evaluation.faithfulness import parse_feature_ranking
from src.utils.results_schema import load_results

FAITHFULNESS_DATASETS = SELECTED_DATASETS
FAITHFULNESS_SEEDS = config.seed_faithfulness

try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed in this environment — skipping pi_self elicitation. "
          "Run this notebook on a GPU box with vllm + the model weights available.")


def best_protocol_for(dataset_name, model_name, baseline_summary):
    """Best non-zero-shot, non-random OOD-accuracy protocol from Notebook 02's
    summary (falls back to 'label_diversity' if Notebook 02 hasn't run yet)."""
    candidates = baseline_summary[
        (baseline_summary.dataset == dataset_name)
        & (baseline_summary.model == model_name)
        & (baseline_summary.environment == 'ood')
        & (~baseline_summary.method.isin(['zero_shot', 'random']))
    ]
    if candidates.empty:
        return 'label_diversity'
    return candidates.sort_values('accuracy_mean', ascending=False).iloc[0]['method']


try:
    baseline_summary = pd.read_parquet(resolve_path('results/real_arm_baselines_summary.parquet'))
except FileNotFoundError:
    baseline_summary = pd.DataFrame(columns=['dataset', 'model', 'method', 'environment', 'accuracy_mean'])

# pi_self doesn't depend on demos or query rows (see the prompt template) so, like
# zero-shot in Notebook 02, it's deterministic (temperature=0) per (dataset, model):
# elicit it once and reuse across the 3 conditions x 3 seeds it's nominally "per".
pi_self_store = {}  # (dataset_name, model_name) -> ranked feature list

for dataset_name in (FAITHFULNESS_DATASETS if VLLM_AVAILABLE else []):
    data_dir = resolve_path(config.paths.data_real) / dataset_name
    feature_list = json.load(open(data_dir / 'feature_list.json'))
    label_tokens = json.load(open(data_dir / 'label_tokens.json'))
    task_description = f"the '{dataset_name}' outcome"
    label_description = f"label {label_tokens[0]} vs {label_tokens[1]}"

    for model_cfg in config.base_llms:
        # VLLMWorkerRunner (not VLLMRunner): runs vLLM in a separate OS
        # subprocess so its GPU memory is guaranteed to be released on
        # shutdown() -- vLLM's in-process engine does not reliably free GPU
        # memory after explicit teardown, which matters here since this loop
        # constructs a fresh runner per (dataset, model) pair.
        runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))
        prompt = build_feature_ranking_prompt(task_description, feature_list, label_description)
        response_text = runner.generate_text([prompt], max_tokens=128)[0]
        ranking = parse_feature_ranking(response_text, feature_list)
        pi_self_store[(dataset_name, model_cfg.name)] = ranking
        print(dataset_name, model_cfg.name, '->', ranking)
        runner.shutdown()

INFO 09-05 14:17:26 [__init__.py:216] Automatically detected platform cuda.
INFO 09-05 14:17:31 [utils.py:233] non-default args: {'max_model_len': 4096, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
WARNING 09-05 14:17:31 [model.py:371] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-05 14:17:32 [model.py:547] Resolved architecture: LlamaForCausalLM
INFO 09-05 14:17:32 [model.py:1510] Using max model len 4096


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 09-05 14:17:33 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-05 14:17:34 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, s

W0905 14:17:38.082000 1780623 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0905 14:17:38.082000 1780623 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 09-05 14:17:40 [parallel_state.py:1208] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
WARNING 09-05 14:17:40 [topk_topp_sampler.py:59] FlashInfer is available, but it is not enabled. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please set VLLM_USE_FLASHINFER_SAMPLER=1.
INFO 09-05 14:17:40 [gpu_model_runner.py:2602] Starting to load model meta-llama/Llama

...

2026-09-05 14:23:19,737 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
2026-09-05 14:23:20,063 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:01<00:00, 37.07it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 67/67 [00:01<00:00, 54.53it/s]


INFO 09-05 14:23:23 [gpu_model_runner.py:3480] Graph capturing finished in 4 secs, took -2.07 GiB
INFO 09-05 14:23:23 [core.py:210] init engine (profile, create kv cache, warmup model) took 12.45 seconds
INFO 09-05 14:23:24 [llm.py:306] Supported_tasks: ('generate',)


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s, est. speed input: 217.93 toks/s, output: 187.21 toks/s]


anes Qwen2.5-7B-Instruct -> ['VCF0301_40', 'VCF0606_00', 'VCF0717_00', 'VCF0717_20', 'VCF0718_00', 'VCF0720_00', 'VCF0721_00', 'VCF0803_90', 'VCF9201_-90', 'VCF9202_-90']


[rank0]:[W905 14:23:26.642902921 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## Step 2: Compute pi_behav via kNN hot-deck LOO ablation

For each feature j: hot-deck-impute it (5 nearest neighbours in the training pool, Euclidean distance on all features except j), re-run inference on the faithfulness subset (200 rows from OOD-test), compute the accuracy drop Delta_j. Rank features by Delta_j descending.

### Why hot-deck imputation, and not just zeroing/masking the feature?

This design choice is a direct application of a principle from **Zhu et al. (2026), "Faithfulness Under the Distribution: A New Look at Attribution Evaluation"** (ICLR 2026), which the lit review cites (ref [29]) specifically for this purpose. That paper's core finding, in the vision domain: standard attribution-evaluation methods (Insertion/Deletion, Infidelity) ablate a feature by zeroing or masking it, which silently introduces new, semantically meaningful evidence rather than removing information — their canonical example is a black-cat-vs-white-cat classifier, where zeroing pixels (making them black) doesn't remove information about "catness," it actively strengthens the "black cat" evidence. The perturbed sample also drifts off the training manifold entirely, and *model behaviour on out-of-distribution inputs is not a reliable signal of the model's real behaviour on the distribution it was trained on*. Using OOD model behaviour to evaluate ID feature importance is, in their words, "highly counterintuitive."

FUD's fix in the vision domain is to use a score-based diffusion model to resynthesise the masked region so it stays on the data manifold. **This project doesn't have (or need) a diffusion model for tabular data** — the equivalent, much cheaper fix for structured features is **hot-deck imputation**: replace the ablated feature's value with a real value sampled from the k=5 nearest neighbours in the training pool (by Euclidean distance on every *other* feature). This keeps the replacement value in-distribution and consistent with the row's other feature values, rather than an artificial zero the model was never trained to see meaningfully. The accuracy drop Δ_j this produces reflects the model's genuine reliance on feature j, not an artefact of showing the model a value it would never encounter naturally.

In [3]:
from tqdm import tqdm

from src.evaluation.faithfulness import hot_deck_impute_feature, compute_accuracy_drop, rank_from_deltas
from src.data.tableshift_loader import select_top_features
from src.data.serialisation import serialise_row, ordered_feature_names
from src.inference.prompts import build_classification_prompt
from src.selection import random_select, similarity_select, label_diversity, feature_range, rule_diversity, counter_spurious
from src.selection.rule_diversity import fit_leaf_tree
from src.selection.counter_spurious import find_spurious_proxy_features

# Same dispatch logic as Notebook 02 — duplicated rather than imported since
# each notebook here is meant to be a self-contained phase of the pipeline.
def prepare_condition_artifacts(dataset_name, train_pool, feature_cols, test_ood):
    artifacts = {}
    continuous_cols = [
        c for c in feature_cols
        if pd.api.types.is_numeric_dtype(train_pool[c]) and train_pool[c].nunique() > 10
    ]
    artifacts['top3_continuous'] = (
        select_top_features(train_pool[continuous_cols + ['label']], n_features=min(3, len(continuous_cols)))
        if continuous_cols else feature_cols[:3]
    )
    artifacts['tree'] = fit_leaf_tree(train_pool, feature_cols)

    # Counter-spurious: proxy feature most correlated with both the label and
    # the actual ID->OOD shift. TableShift's own domain-split covariate (e.g.
    # race/geography/year) is deliberately excluded from X -- it's the exact
    # variable tableshift thresholds to build the ood split, so a model can't
    # just read it directly -- and extract_tableshift_cache.py, which only
    # ever saves X, never had it to cache. Proxy against literal
    # train-vs-OOD-test row membership instead: a feature correlated with
    # *that* carries the same shift signal the raw covariate would have,
    # without needing the excluded column. (Same fix as Notebook 02 -- see
    # that notebook's comment for the full rationale; duplicated here since
    # this notebook is meant to be self-contained.)
    shift_frame = pd.concat(
        [
            train_pool[feature_cols + ['label']].assign(_is_ood=0),
            test_ood[feature_cols + ['label']].assign(_is_ood=1),
        ],
        ignore_index=True,
    )
    proxy_features = find_spurious_proxy_features(shift_frame, feature_cols, 'label', '_is_ood', top_n=3)
    proxy_col = proxy_features[0] if proxy_features else feature_cols[0]
    proxy_high = train_pool[proxy_col] > train_pool[proxy_col].median()
    artifacts['proxy_col'] = proxy_col
    artifacts['proxy_majority_label'] = train_pool.loc[proxy_high, 'label'].mode().iloc[0]
    return artifacts


def select_demos(condition, pool, query, k, seed, feature_cols, artifacts):
    if condition == 'zero_shot':
        return []
    if condition == 'random':
        return random_select.select(pool, query, k, seed)
    if condition == 'label_diversity':
        return label_diversity.select(pool, query, k, seed)
    if condition == 'feature_range':
        return feature_range.select(pool, query, k, seed, top_features=artifacts['top3_continuous'])
    if condition == 'rule_diversity':
        return rule_diversity.select(pool, query, k, seed, feature_cols=feature_cols, tree=artifacts['tree'])
    if condition == 'counter_spurious':
        return counter_spurious.select(
            pool, query, k, seed, proxy_col=artifacts['proxy_col'], proxy_majority_label=artifacts['proxy_majority_label']
        )
    raise ValueError(f"Unknown condition: {condition}")


def build_demo_lines(pool, demo_ids, feature_cols):
    lines = []
    for i in demo_ids:
        row = pool.loc[i]
        ordered = ordered_feature_names({f: row[f] for f in feature_cols})
        lines.append(serialise_row({f: row[f] for f in ordered}, label=str(row['label'])))
    return lines


def build_query_line(query, feature_cols):
    ordered = ordered_feature_names({f: query[f] for f in feature_cols})
    return serialise_row({f: query[f] for f in ordered})


FAITHFULNESS_SUBSET_SEED = FAITHFULNESS_SEEDS[0]
faithfulness_rows = []       # -> results/faithfulness_real.parquet (one row per feature/condition/dataset/seed)
per_row_correct_store = {}   # (dataset, model, condition, seed) -> {feature: bool array}
delta_store = {}             # (dataset, model, condition, seed) -> {feature: delta}

# ~130,000 LLM calls total (see the compute-budget note below) -- the nested
# tqdm bars give a glanceable readout of which (dataset, model, condition,
# seed, feature) is currently running its LOO ablation rerun.
dataset_bar = tqdm(FAITHFULNESS_DATASETS if VLLM_AVAILABLE else [], desc="Datasets", position=0)
for dataset_name in dataset_bar:
    dataset_bar.set_postfix(dataset=dataset_name)
    data_dir = resolve_path(config.paths.data_real) / dataset_name
    train_pool = pd.read_parquet(data_dir / 'train_pool.parquet')
    test_ood_full = pd.read_parquet(data_dir / 'test_ood.parquet')
    feature_cols = json.load(open(data_dir / 'feature_list.json'))
    label_tokens = tuple(json.load(open(data_dir / 'label_tokens.json')))
    task_description = f"the '{dataset_name}' outcome"

    # Fixed across every condition/seed for this dataset, per the spec.
    faith_subset = test_ood_full.sample(
        n=min(config.faithfulness_subset, len(test_ood_full)), random_state=FAITHFULNESS_SUBSET_SEED
    ).reset_index(drop=True)

    artifacts = prepare_condition_artifacts(dataset_name, train_pool, feature_cols, test_ood_full)

    for model_cfg in config.base_llms:
        # VLLMWorkerRunner (not VLLMRunner): runs vLLM in a separate OS
        # subprocess so its GPU memory is guaranteed to be released on
        # shutdown() -- vLLM's in-process engine does not reliably free GPU
        # memory after explicit teardown, which matters here since this loop
        # constructs a fresh runner per (dataset, model) pair.
        runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))
        best_protocol = best_protocol_for(dataset_name, model_cfg.name, baseline_summary)
        conditions_to_eval = ['random', best_protocol]

        # Similarity is deterministic (no seed dependency), so if it's one of
        # this dataset/model's 2 conditions, precompute its demo ids once here
        # (one batched encode() call) rather than inside the seed loop below,
        # which would otherwise re-embed the same faith_subset queries
        # identically on every one of the 3 seeds.
        similarity_demo_ids = None
        if 'similarity' in conditions_to_eval:
            pool_texts = [
                serialise_row(
                    {f: train_pool.loc[i, f] for f in ordered_feature_names({f: train_pool.loc[i, f] for f in feature_cols})},
                    label=str(train_pool.loc[i, 'label']),
                )
                for i in train_pool.index
            ]
            query_texts = [
                serialise_row({f: row[f] for f in ordered_feature_names({f: row[f] for f in feature_cols})})
                for _, row in faith_subset.iterrows()
            ]
            local_idx_per_query = similarity_select.select_batch(pool_texts, query_texts, config.k_primary)
            similarity_demo_ids = [[train_pool.index[i] for i in local_idx] for local_idx in local_idx_per_query]

        condition_bar = tqdm(conditions_to_eval, desc="Conditions", position=1, leave=False)
        for condition in condition_bar:
            condition_bar.set_postfix(condition=condition, model=model_cfg.name)
            seed_bar = tqdm(FAITHFULNESS_SEEDS, desc="Seeds", position=2, leave=False)
            for seed in seed_bar:
                seed_bar.set_postfix(seed=int(seed))
                # Demos are fixed per query across the original run and every feature
                # ablation rerun below, so only the ablated feature can flip a prediction.
                if condition == 'similarity':
                    demo_ids_per_query = similarity_demo_ids
                else:
                    demo_ids_per_query = [
                        select_demos(condition, train_pool, row, config.k_primary, seed, feature_cols, artifacts)
                        for _, row in faith_subset.iterrows()
                    ]

                def run_inference(df):
                    prompts = [
                        build_classification_prompt(
                            task_description, label_tokens,
                            build_demo_lines(train_pool, demo_ids, feature_cols),
                            build_query_line(row, feature_cols),
                        )
                        for (_, row), demo_ids in zip(df.iterrows(), demo_ids_per_query)
                    ]
                    preds = runner.batch_predict(prompts, label_tokens)
                    return np.array([p.prediction == str(row['label']) for p, (_, row) in zip(preds, df.iterrows())])

                original_correct = run_inference(faith_subset)

                deltas, per_row_correct = {}, {}
                feature_bar = tqdm(feature_cols, desc="Features (LOO ablation)", position=3, leave=False)
                for feature in feature_bar:
                    feature_bar.set_postfix(feature=feature)
                    modified = hot_deck_impute_feature(faith_subset, feature, train_pool, feature_cols, seed=seed)
                    modified_correct = run_inference(modified)
                    deltas[feature] = compute_accuracy_drop(original_correct, modified_correct)
                    per_row_correct[feature] = modified_correct

                    faithfulness_rows.append({
                        'dataset': dataset_name, 'model': model_cfg.name, 'method': condition,
                        'seed': int(seed), 'feature': feature, 'delta': deltas[feature],
                    })

                per_row_correct_store[(dataset_name, model_cfg.name, condition, seed)] = per_row_correct
                delta_store[(dataset_name, model_cfg.name, condition, seed)] = deltas
                tqdm.write(f"{dataset_name} {model_cfg.name} {condition} {seed} pi_behav done")

        runner.shutdown()

if not VLLM_AVAILABLE:
    print("Skipped — vLLM not installed in this environment.")

Datasets:   0%|          | 0/4 [00:00<?, ?it/s, dataset=brfss_diabetes]

INFO 09-05 14:23:33 [__init__.py:216] Automatically detected platform cuda.
INFO 09-05 14:23:36 [utils.py:233] non-default args: {'max_model_len': 4096, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
WARNING 09-05 14:23:37 [model.py:371] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-05 14:23:38 [model.py:547] Resolved architecture: LlamaForCausalLM
INFO 09-05 14:23:38 [model.py:1510] Using max model len 4096


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 09-05 14:23:38 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-05 14:23:40 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, s

W0905 14:23:42.721000 1781875 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0905 14:23:42.721000 1781875 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


...

Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 1553.62it/s, est. speed input: 1904145.48 toks/s, output: 1554.36 toks/s]



Features (LOO ablation):   0%|          | 0/10 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2459.97it/s, est. speed input: 3016820.49 toks/s, output: 2462.61 toks/s]



Features (LOO ablation):  10%|█         | 1/10 [00:00<00:05,  1.58it/s, feature=VCF0301_40]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2526.32it/s, est. speed input: 3098264.48 toks/s, output: 2529.10 toks/s]



Features (LOO ablation):  20%|██        | 2/10 [00:01<00:05,  1.58it/s, feature=VCF0606_00]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2512.12it/s, est. speed input: 3080792.44 toks/s, output: 2514.83 toks/s]



Features (LOO ablation):  30%|███       | 3/10 [00:01<00:04,  1.57it/s, feature=VCF0717_00]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2543.29it/s, est. speed input: 3119019.01 toks/s, outp

anes Qwen2.5-7B-Instruct label_diversity 123 pi_behav done


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 1562.96it/s, est. speed input: 1915585.75 toks/s, output: 1563.70 toks/s]



Features (LOO ablation):   0%|          | 0/10 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2511.06it/s, est. speed input: 3079518.36 toks/s, output: 2513.79 toks/s]



Features (LOO ablation):  10%|█         | 1/10 [00:00<00:05,  1.56it/s, feature=VCF0301_40]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2424.98it/s, est. speed input: 2974003.57 toks/s, output: 2427.67 toks/s]



Features (LOO ablation):  20%|██        | 2/10 [00:01<00:06,  1.15it/s, feature=VCF0606_00]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2568.37it/s, est. speed input: 3150006.22 toks/s, output: 2571.32 toks/s]



Features (LOO ablation):  30%|███       | 3/10 [00:02<00:05,  1.31it/s, feature=VCF0717_00]


Processed prompts: 100%|██████████| 200/200 [00:00<00:00, 2480.18it/s, est. speed input: 3041635.77 toks/s, outp

anes Qwen2.5-7B-Instruct label_diversity 456 pi_behav done


[rank0]:[W905 14:35:32.075523414 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
Datasets: 100%|██████████| 4/4 [12:05<00:00, 181.33s/it, dataset=anes]


## Step 3: Spearman rho with bootstrap CIs

ρ(π_self, π_behav) is the headline faithfulness number: a **high** ρ means the model relies on the features it *claims* to rely on (self-report and behaviour agree on ranking); a **low or negative** ρ means the model's stated rationale diverges from what actually drives its predictions — exactly the STaDS-style global unfaithfulness result (Li et al. 2025) this evaluation is designed to detect. Bootstrap CIs (resampling the 200 test rows) give a sense of how much ρ could plausibly vary under a different sample of the same OOD-test distribution, which matters for comparing ρ across conditions (random vs. best-protocol) without over-interpreting small differences.

In [4]:
from src.evaluation.faithfulness import spearman_with_bootstrap

rho_rows = []
for (dataset_name, model_name, condition, seed), deltas in delta_store.items():
    pi_self = pi_self_store[(dataset_name, model_name)]
    per_row_correct = per_row_correct_store[(dataset_name, model_name, condition, seed)]
    result = spearman_with_bootstrap(pi_self, deltas, per_row_correct, n_bootstrap=1000, seed=seed)
    rho_rows.append({
        'dataset': dataset_name, 'model': model_name, 'method': condition, 'seed': int(seed),
        **result,
    })

RHO_PER_SEED_COLS = ['dataset', 'model', 'method', 'seed', 'rho', 'pval', 'ci_low', 'ci_high']
rho_per_seed = pd.DataFrame(rho_rows, columns=RHO_PER_SEED_COLS)

if rho_per_seed.empty:
    print("No faithfulness results yet — skipped (vLLM not available in this environment).")
    rho_summary = pd.DataFrame(columns=['dataset', 'model', 'method', 'rho_mean', 'rho_std', 'ci_low_mean', 'ci_high_mean'])
else:
    # rho +/- CI per (dataset, model, condition): mean/std of the per-seed point
    # estimates, plus the mean of each seed's own bootstrap CI bounds.
    rho_summary = rho_per_seed.groupby(['dataset', 'model', 'method']).agg(
        rho_mean=('rho', 'mean'),
        rho_std=('rho', 'std'),
        ci_low_mean=('ci_low', 'mean'),
        ci_high_mean=('ci_high', 'mean'),
    ).reset_index()

rho_summary

,dataset,model,method,rho_mean,rho_std,ci_low_mean,ci_high_mean
0,acsincome,Llama-3.1-8B-Instruct,label_diversity,0.563636,0.000000,0.563636,0.563636
1,acsincome,Llama-3.1-8B-Instruct,random,0.563636,0.000000,0.563636,0.563636
2,acsincome,Qwen2.5-7B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
3,acsincome,Qwen2.5-7B-Instruct,random,1.000000,0.000000,1.000000,1.000000
4,acspubcov,Llama-3.1-8B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
5,acspubcov,Llama-3.1-8B-Instruct,random,1.000000,0.000000,1.000000,1.000000
6,acspubcov,Qwen2.5-7B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
7,acspubcov,Qwen2.5-7B-Instruct,random,1.000000,0.000000,1.000000,1.000000
8,anes,Llama-3.1-8B-Instruct,label_diversity,0.454545,0.267492,-0.155556,0.749495
9,anes,Llama-3.1-8B-Instruct,random,0.236364,0.515544,0.260606,0.830303


In [5]:
FAITHFULNESS_COLS = ['dataset', 'model', 'method', 'seed', 'feature', 'delta']
faithfulness_df = pd.DataFrame(faithfulness_rows, columns=FAITHFULNESS_COLS)
faithfulness_df.to_parquet(resolve_path('results/faithfulness_real.parquet'), index=False)
rho_per_seed.to_parquet(resolve_path('results/faithfulness_real_rho_per_seed.parquet'), index=False)
rho_summary.to_parquet(resolve_path('results/faithfulness_real_rho_summary.parquet'), index=False)

print(f"faithfulness_real.parquet: {len(faithfulness_df)} rows")
rho_summary

faithfulness_real.parquet: 480 rows


,dataset,model,method,rho_mean,rho_std,ci_low_mean,ci_high_mean
0,acsincome,Llama-3.1-8B-Instruct,label_diversity,0.563636,0.000000,0.563636,0.563636
1,acsincome,Llama-3.1-8B-Instruct,random,0.563636,0.000000,0.563636,0.563636
2,acsincome,Qwen2.5-7B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
3,acsincome,Qwen2.5-7B-Instruct,random,1.000000,0.000000,1.000000,1.000000
4,acspubcov,Llama-3.1-8B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
5,acspubcov,Llama-3.1-8B-Instruct,random,1.000000,0.000000,1.000000,1.000000
6,acspubcov,Qwen2.5-7B-Instruct,label_diversity,1.000000,0.000000,1.000000,1.000000
7,acspubcov,Qwen2.5-7B-Instruct,random,1.000000,0.000000,1.000000,1.000000
8,anes,Llama-3.1-8B-Instruct,label_diversity,0.454545,0.267492,-0.155556,0.749495
9,anes,Llama-3.1-8B-Instruct,random,0.236364,0.515544,0.260606,0.830303


## Compute budget

Per condition: 200 rows x ~12 features x 1 forward pass = 2,400 calls. 3 conditions x 3 seeds x 3 datasets x 2 models ~= 130,000 calls.

## Output

- `results/faithfulness_real.parquet` (one row per feature per condition per dataset per seed)
- Summary: rho +/- CI per (dataset, model, condition)